# Lifecycle

This chapter is about the lifecycle of stream processing with Kafi Streams. It is split into three parts. 

First, we explain how to [set up and build](#setup_build) a processing topology, i.e., how to define the *topology* including *sources* and *sinks* and then how to build it.

Second, we describe the options how to [run and debug](#run_debug) the processing topology both using just the *TopologyNode* class (aka the *test driver*) and the *Streams* class (connects to Kafka).

The third and last section of this chapter is about how to [stop](#stop) the processing.


## Overview

* [Set up and build](#setup_build)
  * [Set up](#setup)
    * [Sources](#sources)
      * [source()](#source)
      * [to_zSet()](#to_zSet)
        * [from_records()](#from_records)
        * [from_debezium()](#from_debezium)
        * [_from_records()](#_from_records)
    * [The topology](#topology)
    * [Sinks](#sinks)
        * [sink()](#sink)
      * [from_zSet()](#from_zSet)
        * [to_records()](#to_records)
        * [to_debezium()](#to_debezium)
        * [_to_records()](#_to_records)
    * [Build](#build)
      * [build()](#build)
      * [reset()](#reset)
* [Run and debug](#run_debug)
  * [Run](#run)
    * [TopologyNode (aka test driver)](#topologynode)
      * [push()](#push)
      * [latest()](#latest)
      * [process()](#process())
    * [Streams](#streams)
      * [start_streams()](#start_streams)
      * [streams()](#streams)
      * [streams_fun()](#streams_fun)
      * [threads()](#threads)
  * [Debug](#debug)
    * [Visualization](#visualization)
      * [topology()](#topology)
      * [mermaid()](#mermaid)
    * [Peeking](#peek)
    * [step_fun (Streams)](#step_fun)
* [Stop](#stop)
  * [stop_fun()](#stop_fun)


---
<a id="building"></a>
## Building

Before you can use a Kafi Streams topology, it needs to be "built". Under the covers, this creates a pydbsp "circuit" that is eventually used for the processing.

`build()` and `reset()` can be used on both `TopologyNode` and `Streams` sink nodes.

<a id="build"></a>
### build()

Central method for building a topology, i.e., wiring it up with the underlying pydbsp library:

```    
@staticmethod
def build(*sink_tn_tuple):
    """Build the circuit for one or more sink nodes.
    
    Args:
        *sink_tn_tuple: one or more sink tn to build
    Returns:
        built_tn: the built topology node"""
```


<a id="reset"></a>
### reset()

With `reset()`, you can rebuild the pydbsp circuit attached to a built topology node from scratch. This also clears all state:

```
def reset(self):
    """Rebuild the circuit from scratch,
    clearing all state."""
```

---
<a id="running"></a>
## Running

This is about running How can I run a Streams thread?

<a id="start_streams"></a>
### start_streams()

Start a Streams thread and get the `stop_fun: None -> None` to stop it:

```
@staticmethod
def start_streams(built_tn, checkpoint_storage=None, checkpoint_topic=None, checkpoint_interval=default_checkpoint_interval_float, **kwargs):
"""Run streams() in a background thread; returns a function to stop it.

Args:
    built_tn: built tn to run
    checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
    checkpoint_topic: topic name used to store checkpoints
    checkpoint_interval: seconds between checkpoints
    **kwargs: passed through to streams()
Returns:
    stop_fun: None -> None function to stop the Streams processing thread"""
```


<a id="streams"></a>
### streams()

Synchronous sub method for `start_streams` - can e.g. be used to run Streams synchronously:

```
@staticmethod
def streams(built_tn, checkpoint_storage=None, checkpoint_topic=None, checkpoint_interval=default_checkpoint_interval_float, stop_event=None, **kwargs):
"""Build producers/consumers from the topology's sources/sinks and run streams_fun().

Args:
    built_tn: built tn to run
    checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
    checkpoint_topic: topic name used to store checkpoints
    checkpoint_interval: seconds between checkpoints
    stop_event: threading.Event that stops the loop once set
    **kwargs: passed through to storage.producer()/consumer()"""
```


<a id="streams_fun"></a>
### streams_fun()

Sub method for `streams()`, actually the main method of Streams. Can be called directly e.g. if you'd like to set up the consumers/producers yourself:

```
@staticmethod
def streams_fun(built_tn, sink_str_foreach_fun_finally_fun_tuple_dict, checkpoint_storage=None, checkpoint_topic=None, checkpoint_interval=default_checkpoint_interval_float, stop_event=None, **kwargs):
    """Main streams loop: consume, push through the topology, produce, checkpoint, repeat.

    Args:
        built_tn: built tn to run
        sink_str_foreach_fun_finally_fun_tuple_dict: dict, sink_str -> (produce_fun, close_fun)
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic: topic name used to store checkpoints
        checkpoint_interval: seconds between checkpoints
        stop_event: threading.Event that stops the loop once set
        **kwargs: extra options, e.g. group, step_fun, progress, chunk_size_bytes"""
```


<a id="threads"></a>
### threads()

```
@staticmethod
def threads():
    """All currently running Streams background threads.
    
    Returns:
        streams_thread_list: the list of currently running Streams threads"""
```
